# 01_split_validation: 分割処理のパターン検証

**目的**: `split.py` の分割処理が、想定される全パターンで期待通りに動作することを目視確認する。

**方針**: 実データ（R001〜R018相当）を反映した20-30件の合成データを使用し、各パターンでの出力を確認する。実データでの検証は、実データSQL接続後に本ノートを拡張する。

## 検証する分割パターン

| カテゴリ | パターン | 期待挙動 |
|---|---|---|
| 通常 | 番号なし | 1レコード（`no_markers`） |
| 通常 | 番号対応 | N分割 |
| 不整合 | 番号セット不一致 | 1レコード（`marker_mismatch`） |
| 不整合 | 番号1つだけ | 1レコード（`no_markers`） |
| 不整合 | 重複マーカー | 1レコード（`duplicate_markers`） |
| 構造 | preamble あり（フリーテキスト） | preamble は context に保持 |
| 構造 | 【】前置き（ユーザ） | 【】は除去、フリーテキストは保持 |
| 構造 | 【】前置き（修理者） | 【】も bracket_prefix に保持 |
| 構造 | 番号項目内【】（ユーザ） | 除去 |
| 構造 | 番号項目内【】（修理者） | 保持 |
| 構造 | postamble（※/▪️等） | postamble カラムに分離 |
| 構造 | 連続マーカー（①②） | 同チャンクを両sub_idに複製 |
| 構造 | 連続+単独混在 | グループごとに適切配分 |
| 例外 | 空レコード | 1レコード（`empty`） |
| 例外 | フォールバック検出 | 警告、不分割 |
| 例外 | 異常分割数 | 警告、分割は実施 |

## セットアップ

In [2]:
import logging
import sys
from pathlib import Path

import pandas as pd

# プロジェクトのsrcをパスに追加
PROJECT_ROOT = Path.cwd().parent  # notebooks/から見たプロジェクトルート
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from split import ColumnMapping, SplitConfig, split_records
from zone_extractor import extract_zones

# 警告ログを表示
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s: %(message)s",
)

# pandas表示オプション
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

print("セットアップ完了")

セットアップ完了


## 1. 合成データの定義

実データ（R001〜R018相当）のパターンを反映した検証用データセット。

In [3]:
TEST_RECORDS = [
    # === 1. 通常レコード（番号なし） ===
    {
        "repair_id": "R001", "product_type": "ML",
        "user_comment": "冬場の屋外撮影でピントが合わなくなります",
        "repair_comment": "低温環境下でのAF動作不良を確認、AFユニット交換にて復旧",
        "internal_1": "同症状増加傾向", "internal_2": "",
    },
    # === 2. きれい分割 ===
    {
        "repair_id": "R002", "product_type": "ML",
        "user_comment": "①AFが効きません ②シャッターも切れません",
        "repair_comment": "①AFユニット交換 ②シャッターブロック交換",
        "internal_1": "保証期間内", "internal_2": "顧客優先",
    },
    # === 3. 3分割 ===
    {
        "repair_id": "R003", "product_type": "LENS",
        "user_comment": "①ズームが固い ②AF迷う ③外装にキズ",
        "repair_comment": "①ズームリング洗浄調整 ②AF調整 ③外装交換",
        "internal_1": "", "internal_2": "",
    },
    # === 4. 番号不一致 ===
    {
        "repair_id": "R004", "product_type": "ML",
        "user_comment": "①ピント不良 ②電源不安定",
        "repair_comment": "①AF調整実施",
        "internal_1": "", "internal_2": "",
    },
    # === 5. 番号1つのみ ===
    {
        "repair_id": "R005", "product_type": "ML",
        "user_comment": "①AF不良",
        "repair_comment": "①AF調整",
        "internal_1": "", "internal_2": "",
    },
    # === 6. フォールバックパターン ===
    {
        "repair_id": "R006", "product_type": "LENS",
        "user_comment": "(1)AF動作不良 (2)異音",
        "repair_comment": "(1)AF調整 (2)レンズユニット内清掃",
        "internal_1": "", "internal_2": "",
    },
    # === 7. 空レコード ===
    {
        "repair_id": "R007", "product_type": "ML",
        "user_comment": "", "repair_comment": "",
        "internal_1": "", "internal_2": "",
    },
    # === 8. 異常分割数 ===
    {
        "repair_id": "R008", "product_type": "ML",
        "user_comment": "①AF ②電源 ③シャッター ④LCD ⑤画像 ⑥音声 ⑦動画 ⑧無線",
        "repair_comment": "①AF調整 ②電池清掃 ③SU交換 ④LCD交換 ⑤センサー清掃 ⑥マイク交換 ⑦FW更新 ⑧基板交換",
        "internal_1": "", "internal_2": "",
    },
    # === 9. 前置き【】+ preamble + postamble (▪️) ===
    {
        "repair_id": "R009", "product_type": "ML",
        "user_comment": "【前回履歴】H111①AF ②電源 ▪️同時預かり",
        "repair_comment": "①AF調整 ②電池清掃 ",
        "internal_1": "", "internal_2": "",
    },
    # === 10. 前置き【】+ postamble (※) ===
    {
        "repair_id": "R010", "product_type": "ML",
        "user_comment": "【前回履歴】H111①AF ②電源 ※同時預かり",
        "repair_comment": "①AF調整 ②電池清掃 ",
        "internal_1": "", "internal_2": "",
    },
    # === 11. 両側postamble ===
    {
        "repair_id": "R011", "product_type": "ML",
        "user_comment": "【前回履歴】H111①AF ②電源 ※同時預かり",
        "repair_comment": "①AF調整 ②電池清掃 ※同時預かり",
        "internal_1": "", "internal_2": "",
    },
    # === 12. ユーザのフリーテキストpreamble ===
    {
        "repair_id": "R012", "product_type": "ML",
        "user_comment": "フリーズ・エラー70・ブラックアウト①エラー ②マウント板ばね破損",
        "repair_comment": "①現象確認できず ②破損確認",
        "internal_1": "", "internal_2": "",
    },
    # === 13. 修理者のフリーテキストpreamble、ユーザは【】 ===
    {
        "repair_id": "R013", "product_type": "ML",
        "user_comment": "【ショック品】【オーバホール】①エラー ②マウント板ばね破損",
        "repair_comment": "オーバーホールを実施いたしました ①現象確認できず ②破損確認",
        "internal_1": "", "internal_2": "",
    },
    # === 14. 番号項目内【】（ユーザは除去） ===
    {
        "repair_id": "R014", "product_type": "ML",
        "user_comment": "①AF 【ご指摘外の現象】②電源 【追加ご指摘】③シャッター 【修理時に発見しました】④外装ラバー劣化",
        "repair_comment": "①AF調整 ②電池清掃 ③シャッター交換 ④外装ラバー交換",
        "internal_1": "", "internal_2": "",
    },
    # === 15. 連続マーカー（修理者まとめ回答） ===
    {
        "repair_id": "R015", "product_type": "ML",
        "user_comment": "①AF ②電源",
        "repair_comment": "①②ご指摘の現象確認",
        "internal_1": "", "internal_2": "",
    },
    # === 16. 連続+単独混在 ===
    {
        "repair_id": "R016", "product_type": "ML",
        "user_comment": "①レンズ接点 ②電源 ③シャッター ④LCD ⑤画像 ⑥音声 ⑦動画",
        "repair_comment": "①②レンズ側にて対応します。カメラには異常なし ③④⑤⑥⑦ご指摘外の現象",
        "internal_1": "", "internal_2": "",
    },
    # === 17. 全部まとめ ===
    {
        "repair_id": "R017", "product_type": "ML",
        "user_comment": "①レンズ接点 ②電源",
        "repair_comment": "①②レンズ側にて対応します。カメラには異常なし ",
        "internal_1": "", "internal_2": "",
    },
    # === 18. 重複マーカー（修理者文中に③が2回） ===
    {
        "repair_id": "R018", "product_type": "ML",
        "user_comment": "メンテを依頼したが、①AF ②電源 ③雨の中で撮影",
        "repair_comment": "①②ご指摘の現象確認できませんでしたが③により発生した可能性が考えられる。③内部の腐食確認",
        "internal_1": "", "internal_2": "",
    },
]

input_df = pd.DataFrame(TEST_RECORDS)
print(f"合成データ件数: {len(input_df)}")
input_df[["repair_id", "product_type", "user_comment", "repair_comment"]]

合成データ件数: 18


,repair_id,product_type,user_comment,repair_comment
0,R001,ML,冬場の屋外撮影でピントが合わなくなります,低温環境下でのAF動作不良を確認、AFユニット交換にて復旧
1,R002,ML,①AFが効きません ②シャッターも切れません,①AFユニット交換 ②シャッターブロック交換
2,R003,LENS,①ズームが固い ②AF迷う ③外装にキズ,①ズームリング洗浄調整 ②AF調整 ③外装交換
3,R004,ML,①ピント不良 ②電源不安定,①AF調整実施
4,R005,ML,①AF不良,①AF調整
5,R006,LENS,(1)AF動作不良 (2)異音,(1)AF調整 (2)レンズユニット内清掃
6,R007,ML,,
7,R008,ML,①AF ②電源 ③シャッター ④LCD ⑤画像 ⑥音声 ⑦動画 ⑧無線,①AF調整 ②電池清掃 ③SU交換 ④LCD交換 ⑤センサー清掃 ⑥マイク交換 ⑦FW更新 ⑧基板交換
8,R009,ML,【前回履歴】H111①AF ②電源 ▪️同時預かり,①AF調整 ②電池清掃
9,R010,ML,【前回履歴】H111①AF ②電源 ※同時預かり,①AF調整 ②電池清掃


## 2. 分割処理の実行

In [4]:
config = SplitConfig()  # デフォルト設定
split_df, report = split_records(input_df, config)

print(report.summary())

分割処理レポート
入力レコード数        : 18
出力レコード数        : 44
  分割なし            : 3
  分割あり            : 12
  番号不一致（不分割） : 1
  重複マーカー（不分割）: 1
  フォールバック検出   : 1
  異常分割数(警告)    : 1
  全カラム空           : 1
警告総数              : 4


## 3. 警告ログの確認

In [5]:
if report.warnings:
    print(f"警告 {len(report.warnings)} 件\n")
    for w in report.warnings:
        print(f"  - {w}")
else:
    print("（警告なし）")

警告 4 件

  - [marker_mismatch] repair_id=R004: ユーザと修理者の番号セットが不一致 (user=[1, 2], repair=[1]) → 分割せず1レコードとして処理
  - [fallback_detected] repair_id=R006: 全角丸数字以外の番号表記を検出 (user=True, repair=True)
  - [abnormal_split] repair_id=R008: 分割数が閾値超過 (8 > 7)
  - [duplicate_markers] repair_id=R018: 同じ番号が複数回出現 (user=False, repair=True) → 分割せず1レコードとして処理


## 4. 出力結果の全体確認

出力DataFrameの全レコードを表示し、想定通りに分割されているか目視確認する。

In [6]:
# サマリー的な表示
split_df[[
    "repair_id", "sub_id", "split_info",
    "user_text", "repair_text",
]]

,repair_id,sub_id,split_info,user_text,repair_text
0,R001,1,no_markers,冬場の屋外撮影でピントが合わなくなります,低温環境下でのAF動作不良を確認、AFユニット交換にて復旧
1,R002,1,split:2,AFが効きません,AFユニット交換
2,R002,2,split:2,シャッターも切れません,シャッターブロック交換
3,R003,1,split:3,ズームが固い,ズームリング洗浄調整
4,R003,2,split:3,AF迷う,AF調整
5,R003,3,split:3,外装にキズ,外装交換
6,R004,1,marker_mismatch,①ピント不良 ②電源不安定,①AF調整実施
7,R005,1,no_markers,①AF不良,①AF調整
8,R006,1,no_markers,(1)AF動作不良 (2)異音,(1)AF調整 (2)レンズユニット内清掃
9,R007,1,empty,,


## 5. ゾーン構造の詳細確認

preamble / context / postamble が正しく抽出されているか、特に重要なケースを抽出して確認。

In [7]:
# preamble/postamble あり のケースだけ抽出
important_ids = ["R009", "R011", "R012", "R013", "R016", "R018"]
subset = split_df[split_df["repair_id"].isin(important_ids)].copy()

for rid in important_ids:
    rows = subset[subset["repair_id"] == rid]
    if rows.empty:
        continue
    print(f"=== {rid} ({rows.iloc[0]['split_info']}) ===")
    for _, row in rows.iterrows():
        print(f"  sub_id={row['sub_id']}")
        print(f"    USER context : {row['user_context']!r}")
        print(f"    USER text    : {row['user_text']!r}")
        print(f"    USER postam. : {row['user_postamble']!r}")
        print(f"    REPAIR ctx   : {row['repair_context']!r}")
        print(f"    REPAIR text  : {row['repair_text']!r}")
        print(f"    REPAIR postam: {row['repair_postamble']!r}")
    print()

=== R009 (split:2) ===
  sub_id=1
    USER context : 'H111'
    USER text    : 'AF'
    USER postam. : '▪️同時預かり'
    REPAIR ctx   : ''
    REPAIR text  : 'AF調整'
    REPAIR postam: ''
  sub_id=2
    USER context : 'H111'
    USER text    : '電源'
    USER postam. : '▪️同時預かり'
    REPAIR ctx   : ''
    REPAIR text  : '電池清掃'
    REPAIR postam: ''

=== R011 (split:2) ===
  sub_id=1
    USER context : 'H111'
    USER text    : 'AF'
    USER postam. : '※同時預かり'
    REPAIR ctx   : ''
    REPAIR text  : 'AF調整'
    REPAIR postam: '※同時預かり'
  sub_id=2
    USER context : 'H111'
    USER text    : '電源'
    USER postam. : '※同時預かり'
    REPAIR ctx   : ''
    REPAIR text  : '電池清掃'
    REPAIR postam: '※同時預かり'

=== R012 (split:2) ===
  sub_id=1
    USER context : 'フリーズ・エラー70・ブラックアウト'
    USER text    : 'エラー'
    USER postam. : ''
    REPAIR ctx   : ''
    REPAIR text  : '現象確認できず'
    REPAIR postam: ''
  sub_id=2
    USER context : 'フリーズ・エラー70・ブラックアウト'
    USER text    : 'マウント板ばね破損'
    USER postam. : ''
    

## 6. パターン別の集計

`split_info` 別の件数を確認し、想定パターンが網羅されているか確認。

In [8]:
split_df.groupby("split_info").size().reset_index(name="count")

,split_info,count
0,duplicate_markers,1
1,empty,1
2,marker_mismatch,1
3,no_markers,3
4,split:2,16
5,split:3,3
6,split:4,4
7,split:7,7
8,split:8,8


## 7. 個別ケースの詳細トレース（zone_extractor 直接呼び出し）

問題のあるケースを発見した際、`zone_extractor` を直接呼び出して詳細を確認する。

In [9]:
# 例: R016 の修理者コメントの詳細を確認
target_text = "①②レンズ側にて対応します。カメラには異常なし ③④⑤⑥⑦ご指摘外の現象"
result = extract_zones(target_text, is_user=False)

print(f"入力: {target_text!r}")
print(f"preamble       : {result.preamble!r}")
print(f"bracket_prefix : {result.bracket_prefix!r}")
print(f"marker_zone    : {result.marker_zone!r}")
print(f"postamble      : {result.postamble!r}")
print(f"has_duplicate  : {result.has_duplicate_markers}")
print(f"\nマーカーグループ:")
for g in result.marker_groups:
    print(f"  {g.marker_ints}: {g.chunk_text!r}")

入力: '①②レンズ側にて対応します。カメラには異常なし ③④⑤⑥⑦ご指摘外の現象'
preamble       : ''
bracket_prefix : ''
marker_zone    : '①②レンズ側にて対応します。カメラには異常なし ③④⑤⑥⑦ご指摘外の現象'
postamble      : ''
has_duplicate  : False

マーカーグループ:
  [1, 2]: 'レンズ側にて対応します。カメラには異常なし'
  [3, 4, 5, 6, 7]: 'ご指摘外の現象'


## 8. 期待結果との比較（assertion）

代表的な分割結果を assertion で検証し、回帰防止する。

In [10]:
# 期待結果
expectations = {
    # repair_id: (sub_id_count, split_info)
    "R001": (1, "no_markers"),
    "R002": (2, "split:2"),
    "R003": (3, "split:3"),
    "R004": (1, "marker_mismatch"),
    "R005": (1, "no_markers"),
    "R006": (1, "no_markers"),       # フォールバックは分割しない
    "R007": (1, "empty"),
    "R008": (8, "split:8"),
    "R009": (2, "split:2"),
    "R010": (2, "split:2"),
    "R011": (2, "split:2"),
    "R012": (2, "split:2"),
    "R013": (2, "split:2"),
    "R014": (4, "split:4"),
    "R015": (2, "split:2"),
    "R016": (7, "split:7"),
    "R017": (2, "split:2"),
    "R018": (1, "duplicate_markers"),
}

all_pass = True
for rid, (expected_count, expected_info) in expectations.items():
    rows = split_df[split_df["repair_id"] == rid]
    actual_count = len(rows)
    actual_info = rows["split_info"].iloc[0] if not rows.empty else None

    status = "✅" if (actual_count == expected_count and actual_info == expected_info) else "❌"
    if status == "❌":
        all_pass = False
        print(f"{status} {rid}: 期待 ({expected_count}件, {expected_info}), 実際 ({actual_count}件, {actual_info})")
    else:
        print(f"{status} {rid}: ({actual_count}件, {actual_info})")

print()
print("全件一致 ✅" if all_pass else "❌ 不一致あり、確認が必要")

✅ R001: (1件, no_markers)
✅ R002: (2件, split:2)
✅ R003: (3件, split:3)
✅ R004: (1件, marker_mismatch)
✅ R005: (1件, no_markers)
✅ R006: (1件, no_markers)
✅ R007: (1件, empty)
✅ R008: (8件, split:8)
✅ R009: (2件, split:2)
✅ R010: (2件, split:2)
✅ R011: (2件, split:2)
✅ R012: (2件, split:2)
✅ R013: (2件, split:2)
✅ R014: (4件, split:4)
✅ R015: (2件, split:2)
✅ R016: (7件, split:7)
✅ R017: (2件, split:2)
✅ R018: (1件, duplicate_markers)

全件一致 ✅


## 9. (オプション) 実データでの検証

実データSQLが接続できるようになったら、ここに実データの検証コードを追加する。

```python
# 例:
# from db import fetch_repair_data
# real_df = fetch_repair_data(limit=100)
# real_split_df, real_report = split_records(real_df)
# print(real_report.summary())
# real_split_df.head(20)
```

## 10. 次のアクション

- 全パターンが期待通りに動作することを確認
- 実データで検証する際は、フォールバック検出件数・重複マーカー件数を監視
- もし新しいパターン（既知のパターンに当てはまらないケース）を発見したら、conftest.pyに追加してテスト化する
- 検証OKであれば、`prompt_builder.py` の実装に進む